Cell 1 — Setup

In [4]:
from pathlib import Path
import hashlib
import json
import subprocess
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
ROOT = Path.home() / "solar_flare_aia"

MANIFEST = ROOT / "training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv"
CACHE_DIR = ROOT / "cache/gcs_npz_canary"
METRICS_DIR = ROOT / "results/metrics"
MODELS_DIR = ROOT / "results/models"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L4


Cell 2 — Load manifest

In [5]:
df = pd.read_csv(MANIFEST, low_memory=False)

print("Rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nYears:")
print(df["year"].value_counts().sort_index())

print("\nLabels:")
print(df["label_48h_final"].value_counts())

print("\nMissing gcp_path:", df["gcp_path"].isna().sum())
print("Missing sample_id:", df["sample_id"].isna().sum())
print("Missing label:", df["label_48h_final"].isna().sum())

Rows: 68010

Columns:
['gcp_path', 'file', 'sample_id', 'T_REC_dt', 'HARPNUM', 'NOAA_AR_clean', 'label_48h_global_old', 'label_48h_ar_specific', 'label_48h_final', 'label_48h', 'y', 'local_path', 'year', 'used_timestamp', 'used_s3_path', 'manifest_label_before_repair']

Years:
year
2010     3306
2011    11142
2012    10815
2013    13066
2014    11627
2015    11236
2016     6818
Name: count, dtype: int64

Labels:
label_48h_final
0    65612
1     2398
Name: count, dtype: int64

Missing gcp_path: 0
Missing sample_id: 0
Missing label: 0


3. Run this first cell

In [2]:
from pathlib import Path
import hashlib
import json
import subprocess
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
ROOT = Path.home() / "solar_flare_aia"

MANIFEST = ROOT / "training/final_metadata/baseline_2010_2016_AR_SPECIFIC_manifest.csv"
CACHE_DIR = ROOT / "cache/gcs_npz_canary"
METRICS_DIR = ROOT / "results/metrics"
MODELS_DIR = ROOT / "results/models"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA L4


4. Then run manifest check cell4. Then run manifest check cell

In [3]:
df = pd.read_csv(MANIFEST, low_memory=False)

print("Rows:", len(df))

print("\nYears:")
print(df["year"].value_counts().sort_index())

print("\nLabels:")
print(df["label_48h_final"].value_counts())

print("\nMissing gcp_path:", df["gcp_path"].isna().sum())
print("Missing sample_id:", df["sample_id"].isna().sum())
print("Missing label:", df["label_48h_final"].isna().sum())

Rows: 68010

Years:
year
2010     3306
2011    11142
2012    10815
2013    13066
2014    11627
2015    11236
2016     6818
Name: count, dtype: int64

Labels:
label_48h_final
0    65612
1     2398
Name: count, dtype: int64

Missing gcp_path: 0
Missing sample_id: 0
Missing label: 0


In [1]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

Torch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA L4


Cell 3 — Choose tiny balanced train/validation set

In [6]:
TRAIN_POS = 24
TRAIN_NEG = 24
VAL_POS = 12
VAL_NEG = 12

def make_balanced_subset(df, years, n_pos, n_neg, seed):
    part = df[df["year"].isin(years)].copy()

    pos = part[part["label_48h_final"] == 1].sample(n_pos, random_state=seed)
    neg = part[part["label_48h_final"] == 0].sample(n_neg, random_state=seed)

    out = pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

train_df = make_balanced_subset(
    df,
    years=[2010, 2011, 2012, 2013],
    n_pos=TRAIN_POS,
    n_neg=TRAIN_NEG,
    seed=SEED,
)

val_df = make_balanced_subset(
    df,
    years=[2014],
    n_pos=VAL_POS,
    n_neg=VAL_NEG,
    seed=SEED + 1,
)

print("Train subset labels:")
print(train_df["label_48h_final"].value_counts())
print("\nTrain years:")
print(train_df["year"].value_counts().sort_index())

print("\nValidation subset labels:")
print(val_df["label_48h_final"].value_counts())
print("\nValidation years:")
print(val_df["year"].value_counts().sort_index())

train_df.to_csv(METRICS_DIR / "canary_aia_cnn_train_samples.csv", index=False)
val_df.to_csv(METRICS_DIR / "canary_aia_cnn_val_samples.csv", index=False)

Train subset labels:
label_48h_final
0    24
1    24
Name: count, dtype: int64

Train years:
year
2010     1
2011    17
2012     9
2013    21
Name: count, dtype: int64

Validation subset labels:
label_48h_final
1    12
0    12
Name: count, dtype: int64

Validation years:
year
2014    24
Name: count, dtype: int64


Cell 4 — Dataset loader

In [7]:
def cache_gcs_file(gcp_path: str) -> Path:
    safe = hashlib.md5(gcp_path.encode()).hexdigest() + ".npz"
    local_path = CACHE_DIR / safe

    if not local_path.exists():
        print(f"Downloading: {gcp_path}")
        subprocess.run(["gcloud", "storage", "cp", gcp_path, str(local_path)], check=True)

    return local_path


class AIANPZDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        local_path = cache_gcs_file(row["gcp_path"])

        data = np.load(local_path, allow_pickle=True)

        # NPZ image is H, W, C = 512, 512, 6
        x = data["x"].astype(np.float32)

        # Convert to PyTorch C, H, W = 6, 512, 512
        x = np.transpose(x, (2, 0, 1))

        # Important: use repaired manifest label, not embedded NPZ y
        y = np.float32(row["label_48h_final"])

        return torch.from_numpy(x), torch.tensor(y), str(row["sample_id"])

Cell 5 — Test one batch before training

In [8]:
BATCH_SIZE = 6

train_loader = DataLoader(
    AIANPZDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    AIANPZDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

x_batch, y_batch, sample_ids = next(iter(train_loader))

print("x batch shape:", x_batch.shape)
print("y batch:", y_batch)
print("sample ids:", sample_ids[:3])
print("x min:", x_batch.min().item())
print("x max:", x_batch.max().item())

Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110926_0836_HARP892_NOAA11302.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110926_0836_HARP892_NOAA11302.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/92a7f0484cf8ca8f2e5f4b601e91a9fb.npz
  
.

Average throughput: 125.9MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120317_1236_HARP1464_NOAA11434.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120317_1236_HARP1464_NOAA11434.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/6b13f39c709f09e3f59ee41017917559.npz
  
.

Average throughput: 87.3MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110307_2112_HARP401_NOAA11166.npz


...

Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110605_1248_HARP637_NOAA11226.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110605_1248_HARP637_NOAA11226.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/7a8d7936d410ee3e86925043f6bb13a0.npz
  
.

Average throughput: 67.2MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120806_1912_HARP1907_NOAA11538.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120806_1912_HARP1907_NOAA11538.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/5b968dbe1d8e0a0d4158ff01f4d44934.npz
  
.

Average throughput: 109.2MiB/s


x batch shape: torch.Size([6, 6, 512, 512])
y batch: tensor([1., 1., 1., 0., 1., 0.])
sample ids: ('20110926_0836_HARP892_NOAA11302', '20120317_1236_HARP1464_NOAA11434', '20110307_2112_HARP401_NOAA11166')
x min: 0.0
x max: 1.0


Cell 6 — Tiny CNN model

In [9]:
class TinyAIACNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


model = TinyAIACNN().to(device)

test_logits = model(x_batch.to(device))
print("Logits shape:", test_logits.shape)
print("Model device:", next(model.parameters()).device)

Logits shape: torch.Size([6])
Model device: cuda:0


Cell 7 — Metrics

In [10]:
def binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    eps = 1e-12

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, eps)
    tss = recall + specificity - 1

    numerator = 2 * (tp * tn - fp * fn)
    denominator = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn))
    hss = numerator / max(denominator, eps)

    return {
        "threshold": threshold,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "tss": tss,
        "hss": hss,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def evaluate(model, loader, device):
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for x, y, _ in loader:
            x = x.to(device)
            logits = model(x)
            prob = torch.sigmoid(logits).cpu().numpy()

            y_prob.extend(prob.tolist())
            y_true.extend(y.numpy().tolist())

    return binary_metrics(y_true, y_prob)

Cell 8 — Train canary

In [11]:
EPOCHS = 2
LR = 1e-3

model = TinyAIACNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []

    for step, (x, y, sample_ids) in enumerate(train_loader, 1):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        losses.append(float(loss.item()))

        if step == 1:
            print(f"Epoch {epoch} first batch x shape:", tuple(x.shape))
            print(f"Epoch {epoch} first batch y:", y.detach().cpu().numpy().tolist())

    val_metrics = evaluate(model, val_loader, device)

    record = {
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        "val_metrics": val_metrics,
    }
    history.append(record)

    print(f"\nEpoch {epoch}/{EPOCHS}")
    print("Train loss:", record["train_loss"])
    print("Val metrics:", val_metrics)

Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120317_1900_HARP1464_NOAA11434.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2012/20120317_1900_HARP1464_NOAA11434.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/56fb880ec97b0b06b4c62bb641152a00.npz
  
.

Average throughput: 71.1MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110905_1336_HARP833_NOAA11283.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110905_1336_HARP833_NOAA11283.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/f8e2cbce12c10b2c0eda8dda920f7523.npz
  
.

Average throughput: 93.6MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2011/20110903_0948_HARP824_NOAA11281.npz


...

Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2014/20140615_1112_HARP4225_NOAA12087.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2014/20140615_1112_HARP4225_NOAA12087.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/7064af53da2649cf4c4caaa86929c88a.npz
  
.

Average throughput: 182.2MiB/s


Downloading: gs://suryabench-sharp-pipeline-bamidele/samples_npz/2014/20141109_0700_HARP4781_NOAA12205.npz


Copying gs://suryabench-sharp-pipeline-bamidele/samples_npz/2014/20141109_0700_HARP4781_NOAA12205.npz to file:///home/abmoses2000/solar_flare_aia/cache/gcs_npz_canary/a259982a7cfc1785cf250d49dabda8f6.npz
  
.

Average throughput: 82.9MiB/s



Epoch 1/2
Train loss: 0.7233648970723152
Val metrics: {'threshold': 0.5, 'accuracy': 0.5, 'precision': 0.5, 'recall': 1.0, 'specificity': 0.0, 'f1': 0.6666666666666666, 'tss': 0.0, 'hss': 0.0, 'tp': 12, 'tn': 0, 'fp': 12, 'fn': 0}
Epoch 2 first batch x shape: (6, 6, 512, 512)
Epoch 2 first batch y: [0.0, 1.0, 1.0, 0.0, 1.0, 0.0]

Epoch 2/2
Train loss: 0.6372788622975349
Val metrics: {'threshold': 0.5, 'accuracy': 0.5, 'precision': 0.5, 'recall': 1.0, 'specificity': 0.0, 'f1': 0.6666666666666666, 'tss': 0.0, 'hss': 0.0, 'tp': 12, 'tn': 0, 'fp': 12, 'fn': 0}
